# ARTEMISIA GLACIALIS v3 — Analyse Opératoire

**Objectif** : Charger les données de trading, diagnostiquer les problèmes, optimiser les paramètres, et améliorer la stratégie dans le temps.

---

## Comment utiliser ce notebook

1. Lancer `dashboard.py` pendant quelques heures (ou une nuit)
2. Le fichier `glacialis3_state.json` est sauvegardé automatiquement
3. Ouvrir ce notebook et exécuter les cellules dans l'ordre
4. Analyser les résultats, ajuster les paramètres, relancer

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from collections import Counter

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['font.size'] = 10

# Couleurs Cyborg
CYAN = '#00e5ff'; GREEN = '#00e676'; RED = '#ff1744'
YELLOW = '#ffd600'; PURPLE = '#d500f9'; TEAL = '#1de9b6'
DIM = '#555555'; BG = '#0a0a0a'

print('✓ Imports OK')

## 1. Chargement des données

In [ ]:
# ══════════════════════════════════════════════
# MODIFIER CE CHEMIN VERS TON FICHIER STATE
# ══════════════════════════════════════════════
STATE_FILE = 'glacialis3_state.json'  # ou chemin complet

with open(STATE_FILE) as f:
    state = json.load(f)

capital = state.get('capital', 500)
peak = state.get('peak', 500)
killed = state.get('killed', False)

# Parse trade log
log = state.get('trade_log', [])
df_log = pd.DataFrame(log)

# Parse equity curve
eq = state.get('equity_curve', [])
df_eq = pd.DataFrame(eq)
if not df_eq.empty and 'timestamp' in df_eq.columns:
    df_eq['timestamp'] = pd.to_datetime(df_eq['timestamp'])

# Séparer OPEN et CLOSE
df_open = df_log[df_log['action'] == 'OPEN'].copy() if not df_log.empty else pd.DataFrame()
df_close = df_log[df_log['action'] == 'CLOSE'].copy() if not df_log.empty else pd.DataFrame()

n_trades = len(df_close)
print(f'Capital: ${capital:.2f} (peak: ${peak:.2f})')
print(f'Killed: {killed}')
print(f'Trade log: {len(df_log)} entries ({len(df_open)} opens, {len(df_close)} closes)')
print(f'Equity curve: {len(df_eq)} points')

## 2. Vue d'ensemble des performances

In [ ]:
if n_trades == 0:
    print('⚠ Aucun trade fermé. Lance la stratégie plus longtemps.')
else:
    pnls = df_close['pnl'].astype(float)
    wins = pnls[pnls > 0]
    losses = pnls[pnls <= 0]
    
    total_pnl = pnls.sum()
    wr = len(wins) / n_trades * 100
    avg_win = wins.mean() if len(wins) > 0 else 0
    avg_loss = losses.mean() if len(losses) > 0 else 0
    pf = wins.sum() / abs(losses.sum()) if losses.sum() != 0 else float('inf')
    fees = df_close['fees'].astype(float).sum()
    
    print('╔══════════════════════════════════════════╗')
    print('║         RÉSUMÉ DE PERFORMANCE             ║')
    print('╠══════════════════════════════════════════╣')
    print(f'║  Trades:      {n_trades:>6}                     ║')
    print(f'║  P&L Total:   ${total_pnl:>+9.2f}                ║')
    print(f'║  Win Rate:    {wr:>6.1f}%                    ║')
    print(f'║  Avg Win:     ${avg_win:>+9.4f}                ║')
    print(f'║  Avg Loss:    ${avg_loss:>+9.4f}                ║')
    print(f'║  Profit Fact: {pf:>6.2f}                     ║')
    print(f'║  Total Fees:  ${fees:>9.4f}                ║')
    print(f'║  Fees/Trade:  ${fees/n_trades:>9.4f}                ║')
    print(f'║  Net/Trade:   ${total_pnl/n_trades:>+9.4f}                ║')
    print('╚══════════════════════════════════════════╝')

## 3. Equity Curve

In [ ]:
if not df_eq.empty:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})
    fig.patch.set_facecolor(BG)
    
    # Equity
    ax1.set_facecolor(BG)
    ax1.plot(df_eq['timestamp'], df_eq['equity'], color=CYAN, linewidth=1.5, label='Equity')
    ax1.axhline(500, color=DIM, linestyle='--', linewidth=0.8, label='Start $500')
    ax1.fill_between(df_eq['timestamp'], 500, df_eq['equity'],
                     where=df_eq['equity'] >= 500, alpha=0.1, color=GREEN)
    ax1.fill_between(df_eq['timestamp'], 500, df_eq['equity'],
                     where=df_eq['equity'] < 500, alpha=0.1, color=RED)
    ax1.set_title('EQUITY CURVE', color=CYAN, fontsize=12, fontweight='bold')
    ax1.set_ylabel('$', color=DIM)
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.1)
    
    # Drawdown
    equity = df_eq['equity'].values
    running_max = np.maximum.accumulate(equity)
    drawdown = (running_max - equity) / running_max * 100
    
    ax2.set_facecolor(BG)
    ax2.fill_between(df_eq['timestamp'], 0, -drawdown, color=RED, alpha=0.4)
    ax2.plot(df_eq['timestamp'], -drawdown, color=RED, linewidth=0.8)
    ax2.set_ylabel('Drawdown %', color=DIM)
    ax2.set_title('DRAWDOWN', color=RED, fontsize=10)
    ax2.grid(alpha=0.1)
    
    plt.tight_layout()
    plt.show()
    
    print(f'Max Drawdown: {drawdown.max():.2f}%')
    print(f'Max Drawdown $: ${(running_max - equity).max():.2f}')
else:
    print('Pas de données equity curve')

## 4. Analyse par Symbole — Quel coin gagne/perd ?

In [ ]:
if n_trades > 0:
    sym_stats = df_close.groupby('symbol').agg(
        trades=('pnl', 'count'),
        total_pnl=('pnl', lambda x: x.astype(float).sum()),
        avg_pnl=('pnl', lambda x: x.astype(float).mean()),
        win_rate=('pnl', lambda x: (x.astype(float) > 0).mean() * 100),
        total_fees=('fees', lambda x: x.astype(float).sum()),
    ).sort_values('total_pnl', ascending=True)
    
    print('\nPerformance par symbole:')
    print('=' * 75)
    print(f'{"Symbol":>8} {"Trades":>7} {"P&L":>10} {"Avg":>10} {"WR%":>6} {"Fees":>8} {"Net/Trade":>10}')
    print('-' * 75)
    for sym, row in sym_stats.iterrows():
        net_per = (row['total_pnl']) / row['trades'] if row['trades'] > 0 else 0
        color = '🟢' if row['total_pnl'] > 0 else '🔴'
        print(f"{color} {sym:>6} {row['trades']:>7} ${row['total_pnl']:>+9.4f} "
              f"${row['avg_pnl']:>+9.4f} {row['win_rate']:>5.1f}% "
              f"${row['total_fees']:>7.4f} ${net_per:>+9.4f}")
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    colors = [GREEN if v > 0 else RED for v in sym_stats['total_pnl']]
    ax.barh(sym_stats.index, sym_stats['total_pnl'], color=colors)
    ax.axvline(0, color=DIM, linewidth=0.5)
    ax.set_title('P&L PAR SYMBOLE', color=CYAN, fontsize=12, fontweight='bold')
    ax.set_xlabel('P&L ($)', color=DIM)
    for i, (sym, row) in enumerate(sym_stats.iterrows()):
        ax.text(row['total_pnl'], i, f" ${row['total_pnl']:+.3f} ({row['trades']}t)",
                va='center', fontsize=8, color=DIM)
    ax.grid(alpha=0.1)
    plt.tight_layout()
    plt.show()
    
    # DIAGNOSTIC: symboles à blacklister
    print('\n⚠ SYMBOLES À SURVEILLER (>5 trades et WR < 40%):')
    for sym, row in sym_stats.iterrows():
        if row['trades'] >= 5 and row['win_rate'] < 40:
            print(f'  🔴 {sym}: {row["trades"]} trades, WR={row["win_rate"]:.0f}%, '
                  f'P&L=${row["total_pnl"]:+.4f} → envisager de blacklister')

## 5. Analyse par Stratégie (MR / MB / SP)

In [ ]:
if n_trades > 0 and 'strat' in df_close.columns:
    strat_stats = df_close.groupby('strat').agg(
        trades=('pnl', 'count'),
        total_pnl=('pnl', lambda x: x.astype(float).sum()),
        avg_pnl=('pnl', lambda x: x.astype(float).mean()),
        win_rate=('pnl', lambda x: (x.astype(float) > 0).mean() * 100),
        avg_hold=('hold_sec', lambda x: x.astype(float).mean() if 'hold_sec' in df_close.columns else 0),
    )
    
    print('Performance par stratégie:')
    print('=' * 65)
    for strat, row in strat_stats.iterrows():
        icon = '🟢' if row['total_pnl'] > 0 else '🔴'
        name = {'MR': 'Mean Reversion', 'MB': 'Momentum Burst', 'SP': 'Spread Reversion'}.get(strat, strat)
        print(f'\n{icon} {strat} — {name}')
        print(f'  Trades: {row["trades"]:>5}  |  P&L: ${row["total_pnl"]:+.4f}  |  '
              f'WR: {row["win_rate"]:.1f}%  |  Avg: ${row["avg_pnl"]:+.4f}  |  '
              f'Hold: {row["avg_hold"]:.1f}s')
    
    # Recommandation
    print('\n\n📋 RECOMMANDATIONS:')
    for strat, row in strat_stats.iterrows():
        if row['trades'] >= 10 and row['win_rate'] < 45:
            print(f'  ⚠ {strat}: WR {row["win_rate"]:.0f}% < 45% → envisager de désactiver')
        elif row['trades'] >= 10 and row['total_pnl'] < 0:
            print(f'  ⚠ {strat}: P&L négatif → resserrer les seuils d\'entrée')
        elif row['trades'] >= 10 and row['win_rate'] > 55:
            print(f'  ✓ {strat}: WR {row["win_rate"]:.0f}% > 55% → potentiel pour relâcher les seuils')
        elif row['trades'] < 5:
            print(f'  ℹ {strat}: seulement {row["trades"]} trades → pas assez pour juger')

## 6. Distribution des P&L et Hold Times

In [ ]:
if n_trades > 5:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.patch.set_facecolor(BG)
    
    pnls = df_close['pnl'].astype(float)
    
    # P&L distribution
    ax = axes[0]
    ax.set_facecolor(BG)
    ax.hist(pnls, bins=min(50, n_trades//2), color=CYAN, alpha=0.7, edgecolor='none')
    ax.axvline(0, color=RED, linestyle='--', linewidth=1)
    ax.axvline(pnls.mean(), color=YELLOW, linestyle='-', linewidth=1.5, label=f'Mean: ${pnls.mean():.4f}')
    ax.set_title('DISTRIBUTION P&L', color=CYAN, fontsize=10)
    ax.set_xlabel('P&L ($)', color=DIM)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.1)
    
    # Hold time distribution
    ax = axes[1]
    ax.set_facecolor(BG)
    if 'hold_sec' in df_close.columns:
        holds = df_close['hold_sec'].astype(float)
        ax.hist(holds, bins=min(40, n_trades//2), color=TEAL, alpha=0.7, edgecolor='none')
        ax.axvline(holds.mean(), color=YELLOW, linestyle='-', linewidth=1.5,
                   label=f'Mean: {holds.mean():.1f}s')
        ax.set_xlabel('Hold Time (s)', color=DIM)
        ax.legend(fontsize=8)
    ax.set_title('DISTRIBUTION HOLD TIME', color=TEAL, fontsize=10)
    ax.grid(alpha=0.1)
    
    # P&L cumulé
    ax = axes[2]
    ax.set_facecolor(BG)
    cumul = pnls.cumsum()
    ax.plot(range(len(cumul)), cumul, color=CYAN, linewidth=1.5)
    ax.fill_between(range(len(cumul)), 0, cumul,
                    where=cumul >= 0, alpha=0.1, color=GREEN)
    ax.fill_between(range(len(cumul)), 0, cumul,
                    where=cumul < 0, alpha=0.1, color=RED)
    ax.axhline(0, color=DIM, linewidth=0.5)
    ax.set_title('P&L CUMULÉ', color=CYAN, fontsize=10)
    ax.set_xlabel('Trade #', color=DIM)
    ax.set_ylabel('$', color=DIM)
    ax.grid(alpha=0.1)
    
    plt.tight_layout()
    plt.show()
    
    # Stats détaillées
    print(f'P&L: median=${pnls.median():.4f}  std=${pnls.std():.4f}  '
          f'skew={pnls.skew():.2f}  kurtosis={pnls.kurtosis():.2f}')
    if 'hold_sec' in df_close.columns:
        print(f'Hold: median={holds.median():.1f}s  max={holds.max():.1f}s  '
              f'min={holds.min():.1f}s')

## 7. Analyse des Exit Reasons — Pourquoi les trades se ferment ?

In [ ]:
if n_trades > 0 and 'exit_reason' in df_close.columns:
    reason_stats = df_close.groupby('exit_reason').agg(
        count=('pnl', 'count'),
        pnl=('pnl', lambda x: x.astype(float).sum()),
        avg_pnl=('pnl', lambda x: x.astype(float).mean()),
        wr=('pnl', lambda x: (x.astype(float) > 0).mean() * 100),
    ).sort_values('count', ascending=False)
    
    print('Exit Reasons:')
    print('=' * 65)
    print(f'{"Reason":<15} {"Count":>6} {"% Total":>8} {"P&L":>10} {"Avg":>10} {"WR%":>6}')
    print('-' * 65)
    for reason, row in reason_stats.iterrows():
        pct = row['count'] / n_trades * 100
        icon = '🟢' if row['pnl'] > 0 else '🔴'
        print(f"{icon} {reason:<13} {row['count']:>6} {pct:>7.1f}% "
              f"${row['pnl']:>+9.4f} ${row['avg_pnl']:>+9.4f} {row['wr']:>5.1f}%")
    
    # DIAGNOSTIC
    print('\n\n📋 DIAGNOSTIC EXIT REASONS:')
    if 'STOP' in reason_stats.index:
        stop_pct = reason_stats.loc['STOP', 'count'] / n_trades * 100
        if stop_pct > 50:
            print(f'  ⚠ {stop_pct:.0f}% des trades se ferment sur STOP')
            print(f'    → Le stop est probablement trop serré. Augmenter stop_n_sigma (actuel: 4.0)')
            print(f'    → Ou les seuils d\'entrée sont trop faibles (bruit ≠ signal)')
        elif stop_pct < 15:
            print(f'  ✓ Seulement {stop_pct:.0f}% de stops — bon signe')
    
    if 'MAX_HOLD' in reason_stats.index:
        mh_pct = reason_stats.loc['MAX_HOLD', 'count'] / n_trades * 100
        mh_wr = reason_stats.loc['MAX_HOLD', 'wr']
        if mh_pct > 30:
            print(f'  ⚠ {mh_pct:.0f}% des trades atteignent MAX_HOLD (WR={mh_wr:.0f}%)')
            print(f'    → La reversion est plus lente que prévu')
            print(f'    → Augmenter max_hold ou augmenter mr_max_half_life')
    
    for reason in ['MR_REVERT', 'SP_REVERT', 'MB_CAUGHT']:
        if reason in reason_stats.index:
            wr = reason_stats.loc[reason, 'wr']
            print(f'  ✓ {reason}: WR={wr:.0f}% — '
                  f'{"excellent" if wr > 60 else "correct" if wr > 50 else "à surveiller"}')

## 8. Analyse temporelle — Quand la strat marche le mieux ?

In [ ]:
if n_trades > 10 and 'ts_full' in df_close.columns:
    df_close['datetime'] = pd.to_datetime(df_close['ts_full'], errors='coerce')
    df_close['hour'] = df_close['datetime'].dt.hour
    
    hourly = df_close.groupby('hour').agg(
        trades=('pnl', 'count'),
        pnl=('pnl', lambda x: x.astype(float).sum()),
        wr=('pnl', lambda x: (x.astype(float) > 0).mean() * 100),
    )
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    fig.patch.set_facecolor(BG)
    
    ax1.set_facecolor(BG)
    colors = [GREEN if v > 0 else RED for v in hourly['pnl']]
    ax1.bar(hourly.index, hourly['pnl'], color=colors, alpha=0.8)
    ax1.set_title('P&L PAR HEURE (UTC)', color=CYAN, fontsize=10)
    ax1.set_xlabel('Heure UTC', color=DIM)
    ax1.set_ylabel('P&L ($)', color=DIM)
    ax1.grid(alpha=0.1)
    
    ax2.set_facecolor(BG)
    ax2.bar(hourly.index, hourly['trades'], color=TEAL, alpha=0.7)
    ax2.set_title('NOMBRE DE TRADES PAR HEURE', color=TEAL, fontsize=10)
    ax2.set_xlabel('Heure UTC', color=DIM)
    ax2.grid(alpha=0.1)
    
    plt.tight_layout()
    plt.show()
    
    best_hour = hourly['pnl'].idxmax()
    worst_hour = hourly['pnl'].idxmin()
    print(f'Meilleure heure: {best_hour}:00 UTC (P&L: ${hourly.loc[best_hour, "pnl"]:+.4f})')
    print(f'Pire heure: {worst_hour}:00 UTC (P&L: ${hourly.loc[worst_hour, "pnl"]:+.4f})')

## 9. Séries gagnantes/perdantes — Détection de tilt

In [ ]:
if n_trades > 5:
    pnls = df_close['pnl'].astype(float).values
    is_win = pnls > 0
    
    # Séries consécutives
    streaks = []
    current = 1
    for i in range(1, len(is_win)):
        if is_win[i] == is_win[i-1]:
            current += 1
        else:
            streaks.append(('W' if is_win[i-1] else 'L', current))
            current = 1
    streaks.append(('W' if is_win[-1] else 'L', current))
    
    w_streaks = [s[1] for s in streaks if s[0] == 'W']
    l_streaks = [s[1] for s in streaks if s[0] == 'L']
    
    print(f'Max série gagnante: {max(w_streaks) if w_streaks else 0}')
    print(f'Max série perdante: {max(l_streaks) if l_streaks else 0}')
    print(f'Avg série gagnante: {np.mean(w_streaks):.1f}' if w_streaks else '')
    print(f'Avg série perdante: {np.mean(l_streaks):.1f}' if l_streaks else '')
    
    if l_streaks and max(l_streaks) > 8:
        print(f'\n⚠ ALERTE: série de {max(l_streaks)} pertes consécutives')
        print(f'  → Vérifier si c\'est sur le même symbole (churn)')
        print(f'  → Augmenter cooldown_after_loss')
    
    # Rolling win rate (fenêtre de 20 trades)
    if n_trades >= 20:
        window = 20
        rolling_wr = pd.Series(is_win).rolling(window).mean() * 100
        
        fig, ax = plt.subplots(figsize=(14, 4))
        fig.patch.set_facecolor(BG)
        ax.set_facecolor(BG)
        ax.plot(range(len(rolling_wr)), rolling_wr, color=CYAN, linewidth=1.5)
        ax.axhline(50, color=YELLOW, linestyle='--', linewidth=1, label='50% (seuil)')
        ax.fill_between(range(len(rolling_wr)), 50, rolling_wr,
                        where=rolling_wr >= 50, alpha=0.15, color=GREEN)
        ax.fill_between(range(len(rolling_wr)), 50, rolling_wr,
                        where=rolling_wr < 50, alpha=0.15, color=RED)
        ax.set_title(f'WIN RATE GLISSANT ({window} trades)', color=CYAN, fontsize=10)
        ax.set_xlabel('Trade #', color=DIM)
        ax.set_ylabel('Win Rate %', color=DIM)
        ax.legend(fontsize=8)
        ax.set_ylim(0, 100)
        ax.grid(alpha=0.1)
        plt.tight_layout()
        plt.show()

## 10. Analyse Edge vs Résultat — Les signaux sont-ils calibrés ?

In [ ]:
if n_trades > 10 and 'edge_bps' in df_open.columns:
    # Merger open et close pour avoir edge ET pnl
    opens = df_open[['symbol', 'edge_bps', 'hurst', 'hl', 'vol']].copy()
    opens = opens.reset_index(drop=True)
    closes = df_close[['symbol', 'pnl', 'pnl_pct', 'exit_reason', 'hold_sec']].copy()
    closes = closes.reset_index(drop=True)
    
    if len(opens) == len(closes):
        merged = pd.concat([opens, closes[['pnl', 'pnl_pct', 'exit_reason', 'hold_sec']]], axis=1)
        merged['pnl'] = merged['pnl'].astype(float)
        merged['edge_bps'] = merged['edge_bps'].astype(float)
        merged['won'] = merged['pnl'] > 0
        
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.patch.set_facecolor(BG)
        
        # Edge vs P&L scatter
        ax = axes[0]
        ax.set_facecolor(BG)
        colors = [GREEN if w else RED for w in merged['won']]
        ax.scatter(merged['edge_bps'], merged['pnl'], c=colors, alpha=0.5, s=15)
        ax.axhline(0, color=DIM, linewidth=0.5)
        ax.set_title('EDGE PRÉDIT vs P&L RÉEL', color=CYAN, fontsize=10)
        ax.set_xlabel('Edge prédit (bps)', color=DIM)
        ax.set_ylabel('P&L réel ($)', color=DIM)
        ax.grid(alpha=0.1)
        
        # Hurst vs WR
        ax = axes[1]
        ax.set_facecolor(BG)
        if 'hurst' in merged.columns:
            merged['hurst'] = merged['hurst'].astype(float)
            bins = pd.cut(merged['hurst'], bins=5)
            hurst_wr = merged.groupby(bins)['won'].mean() * 100
            ax.bar(range(len(hurst_wr)), hurst_wr.values, color=PURPLE, alpha=0.7)
            ax.set_xticks(range(len(hurst_wr)))
            ax.set_xticklabels([f'{b.left:.2f}' for b in hurst_wr.index], fontsize=7)
            ax.axhline(50, color=YELLOW, linestyle='--', linewidth=1)
        ax.set_title('WIN RATE PAR HURST', color=PURPLE, fontsize=10)
        ax.set_xlabel('Hurst Exponent', color=DIM)
        ax.set_ylabel('Win Rate %', color=DIM)
        ax.grid(alpha=0.1)
        
        # Edge bins vs Win Rate
        ax = axes[2]
        ax.set_facecolor(BG)
        edge_bins = pd.cut(merged['edge_bps'], bins=5)
        edge_wr = merged.groupby(edge_bins)['won'].mean() * 100
        ax.bar(range(len(edge_wr)), edge_wr.values, color=TEAL, alpha=0.7)
        ax.set_xticks(range(len(edge_wr)))
        ax.set_xticklabels([f'{b.left:.0f}' for b in edge_wr.index], fontsize=7)
        ax.axhline(50, color=YELLOW, linestyle='--', linewidth=1)
        ax.set_title('WIN RATE PAR EDGE', color=TEAL, fontsize=10)
        ax.set_xlabel('Edge (bps)', color=DIM)
        ax.set_ylabel('Win Rate %', color=DIM)
        ax.grid(alpha=0.1)
        
        plt.tight_layout()
        plt.show()
        
        # Corrélation edge-pnl
        corr = merged['edge_bps'].corr(merged['pnl'])
        print(f'\nCorrélation Edge → P&L: {corr:.3f}')
        if corr > 0.1:
            print('  ✓ Le signal edge prédit correctement les résultats')
        elif corr < -0.05:
            print('  ⚠ Le signal edge est inversement corrélé → problème de modèle')
        else:
            print('  ℹ Corrélation faible → le signal edge n\'est pas très prédictif')

## 11. Fees Analysis — Les frais mangent-ils l'alpha ?

In [ ]:
if n_trades > 0:
    pnls = df_close['pnl'].astype(float)
    fees = df_close['fees'].astype(float)
    gross_pnl = pnls + fees  # P&L avant fees
    
    print('Analyse des frais:')
    print(f'  P&L brut (avant fees):  ${gross_pnl.sum():+.4f}')
    print(f'  Total fees:             ${fees.sum():.4f}')
    print(f'  P&L net (après fees):   ${pnls.sum():+.4f}')
    print(f'  Fees / P&L brut:        {fees.sum() / abs(gross_pnl.sum()) * 100:.1f}%' if gross_pnl.sum() != 0 else '')
    print(f'  Fee moyenne par trade:  ${fees.mean():.4f}')
    
    # Est-ce que la stratégie serait rentable sans fees ?
    if gross_pnl.sum() > 0 and pnls.sum() < 0:
        print(f'\n  ⚠ STRATÉGIE PROFITABLE AVANT FEES MAIS PAS APRÈS')
        print(f'    → L\'alpha brut existe (${gross_pnl.sum():+.4f}) mais les fees l\'effacent')
        print(f'    → Solutions: augmenter min_edge_bps, ou réduire le nombre de trades')
    elif gross_pnl.sum() < 0:
        print(f'\n  🔴 STRATÉGIE NON PROFITABLE MÊME AVANT FEES')
        print(f'    → Le signal n\'a pas d\'edge réel. Revoir les seuils et filtres.')
    elif pnls.sum() > 0:
        print(f'\n  ✓ STRATÉGIE PROFITABLE APRÈS FEES')
        fee_drag = fees.sum() / gross_pnl.sum() * 100
        print(f'    → Les fees consomment {fee_drag:.0f}% de l\'alpha brut')

## 12. Recommandations de paramètres

In [ ]:
if n_trades >= 20:
    pnls = df_close['pnl'].astype(float)
    wr = (pnls > 0).mean() * 100
    
    print('╔══════════════════════════════════════════════════════════╗')
    print('║              RECOMMANDATIONS PARAMÈTRES                  ║')
    print('╠══════════════════════════════════════════════════════════╣')
    
    # Win rate analysis
    if wr < 40:
        print('║  ⚠ Win Rate < 40% — Problème fondamental               ║')
        print('║    → Augmenter mr_entry_sigma: 2.5 → 3.0               ║')
        print('║    → Baisser mr_max_hurst: 0.45 → 0.40                 ║')
        print('║    → Augmenter min_edge_bps: 10 → 15                   ║')
        print('║    → Augmenter stop_n_sigma: 4.0 → 5.0                 ║')
    elif wr < 50:
        print('║  ℹ Win Rate 40-50% — Marginal                          ║')
        print('║    → Léger resserrement des seuils                      ║')
        print('║    → Augmenter min_edge_bps: 10 → 12                   ║')
    elif wr < 55:
        print('║  ✓ Win Rate 50-55% — Correct                           ║')
        print('║    → Garder les paramètres actuels                      ║')
    else:
        print('║  ✓ Win Rate > 55% — Excellent                          ║')
        print('║    → Potentiel pour relâcher les seuils :               ║')
        print('║      mr_entry_sigma: 2.5 → 2.0                         ║')
        print('║      min_edge_bps: 10 → 8                              ║')
    
    # Stop analysis
    if 'exit_reason' in df_close.columns:
        stop_ratio = (df_close['exit_reason'] == 'STOP').mean() * 100
        if stop_ratio > 40:
            print('║                                                        ║')
            print(f'║  ⚠ {stop_ratio:.0f}% de STOPS — trop fréquent                  ║')
            print('║    → Augmenter stop_n_sigma: 4.0 → 5.0                 ║')
    
    # Trade frequency
    if 'ts_full' in df_close.columns:
        df_close['dt'] = pd.to_datetime(df_close['ts_full'], errors='coerce')
        valid_dt = df_close['dt'].dropna()
        if len(valid_dt) > 1:
            span_hours = (valid_dt.max() - valid_dt.min()).total_seconds() / 3600
            tph = n_trades / max(span_hours, 0.1)
            print(f'║                                                        ║')
            print(f'║  Fréquence: {tph:.1f} trades/heure sur {span_hours:.1f}h        ║')
            if tph > 30:
                print('║    → Trop fréquent. Augmenter cooldown ou seuils.      ║')
            elif tph < 2:
                print('║    → Peu fréquent. Baisser seuils si WR est bon.       ║')
    
    print('╚══════════════════════════════════════════════════════════╝')
else:
    print(f'Seulement {n_trades} trades — il en faut au moins 20 pour des recommandations fiables.')
    print('Lance la stratégie plus longtemps et reviens.')

## 13. Export des trades pour analyse externe

In [ ]:
if n_trades > 0:
    # Export CSV
    export_path = 'glacialis_trades_export.csv'
    df_close.to_csv(export_path, index=False)
    print(f'✓ Trades exportés vers {export_path}')
    print(f'  {n_trades} trades, colonnes: {list(df_close.columns)}')
    
    # Aperçu
    print('\nDerniers 10 trades:')
    cols = ['ts', 'symbol', 'side', 'strat', 'price', 'pnl', 'pnl_pct', 'exit_reason', 'hold_sec']
    available_cols = [c for c in cols if c in df_close.columns]
    print(df_close[available_cols].tail(10).to_string(index=False))

---

## Checklist d'amélioration continue

À chaque session d'analyse, vérifier :

1. **Win Rate global** : > 50% = OK, < 45% = resserrer
2. **Win Rate par stratégie** : désactiver celles qui perdent
3. **Win Rate par symbole** : blacklister les perdants chroniques
4. **Ratio STOP / total** : si > 40%, élargir les stops
5. **Corrélation Edge → P&L** : si < 0, le signal ne marche pas
6. **Fees / P&L brut** : si > 80%, l'alpha est trop faible
7. **Max série perdante** : si > 10, augmenter cooldown
8. **Rolling WR** : si constamment < 50%, stop la strat